In [ ]:
from pathlib import Path
import zipfile


ZIP_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\instuments")

RAW_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets/raw_unzipped")
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets/yolo_dataset")
(OUT_DIR / "images/train").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "images/val").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "labels/train").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "labels/val").mkdir(parents=True, exist_ok=True)

 
for z in ZIP_DIR.glob("*.zip"):
    cls_name = z.stem   
    cls_dir = RAW_DIR / cls_name
    cls_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(z, "r") as zip_ref:
        zip_ref.extractall(cls_dir)

print("✅ Распаковали ZIP-архивы")
print("Найденные классы:", [p.name for p in RAW_DIR.iterdir() if p.is_dir()])

# === список классов ===
CLASSES = sorted([p.name for p in RAW_DIR.iterdir() if p.is_dir()])
CLASS2ID = {c: i for i, c in enumerate(CLASSES)}

print("Классы для обучения:", CLASSES)


✅ Распаковали ZIP-архивы
Найденные классы: ['1 Отвертка «-»', '10 Ключ рожковыйнакидной  ¾', '11 Бокорезы', '2 Отвертка «+»', '3 Отвертка на смещенный крест', '4 Коловорот', '5 Пассатижи контровочные', '6 Пассатижи', '7 Шэрница', '8 Разводной ключ', '9 Открывашка для банок с маслом']
Классы для обучения: ['1 Отвертка «-»', '10 Ключ рожковыйнакидной  ¾', '11 Бокорезы', '2 Отвертка «+»', '3 Отвертка на смещенный крест', '4 Коловорот', '5 Пассатижи контровочные', '6 Пассатижи', '7 Шэрница', '8 Разводной ключ', '9 Открывашка для банок с маслом']


In [ ]:
from pathlib import Path
import zipfile
ZIP_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\instuments")

 
RAW_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets/raw_unzipped")
RAW_DIR.mkdir(parents=True, exist_ok=True)

 
OUT_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets/yolo_dataset")
CLASSES = sorted([p.name for p in RAW_DIR.iterdir() if p.is_dir()])
CLASS2ID = {c: i for i, c in enumerate(CLASSES)}

In [ ]:
import torch
from transformers import pipeline
 
device = 0 if torch.cuda.is_available() else -1
print("Device:", "CUDA" if device == 0 else "CPU")
 
detector = pipeline(
    task="zero-shot-object-detection",
    model="IDEA-Research/grounding-dino-base",
    device=device
)
 
candidate_labels = [
    "screwdriver flat", "screwdriver cross", "screwdriver offset",
    "brace", "safety pliers", "pliers", "shernitsa",
    "adjustable wrench", "oil can opener",
    "combination wrench 3/4", "side cutters"
]


Device: CUDA


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


In [8]:
import torch
device = 0 if torch.cuda.is_available() else -1


def get_best_box(result, W, H):
    if result is None or len(result) == 0:
        return 0, 0, W, H

    best_score = -1
    best_box = None
    for r in result:
        x1, y1, x2, y2 = r["box"].values()
        w = x2 - x1
        h = y2 - y1
        area = (w * h) / (W * H)

        if area < 0.01 or area > 0.8:
            continue

        if r["score"] > best_score:
            best_score = r["score"]
            best_box = (x1, y1, w, h)

    if best_box is None:
        r = max(result, key=lambda x: x["score"])
        x1, y1, x2, y2 = r["box"].values()
        best_box = (x1, y1, x2 - x1, y2 - y1)

    return best_box


def yolo_line(cls_id, x,y,w,h, W,H):
    cx = (x + w/2)/W
    cy = (y + h/2)/H
    nw = w/W
    nh = h/H
    return f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n"

In [ ]:
# -----Предразметка с помощью Dino-----

import random
from tqdm import tqdm
import cv2
import shutil
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

import random
from tqdm import tqdm
import cv2
import shutil
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".JPG", ".JPEG", ".PNG"}
VAL_SPLIT = 0.15
PREVIEW_EVERY = 50


PREVIEW_DIR = OUT_DIR / "preview"
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)


def extract_class_id_from_folder(folder_name: str) -> int:
    """Если папка начинается с числа (до пробела) → используем это число.
       Иначе берём индекс из CLASS2ID."""
    parts = folder_name.split()
    if parts and parts[0].isdigit():
        return int(parts[0])
    return CLASS2ID[folder_name]


for cls in CLASSES:
    paths = [p for p in (RAW_DIR / cls).glob("**/*") if p.suffix in IMG_EXTS]
    random.shuffle(paths)
    n_val = max(1, int(len(paths) * VAL_SPLIT))

    for i, p in enumerate(tqdm(paths, desc=cls)):
        subset = "val" if i < n_val else "train"
        out_img = OUT_DIR / f"images/{subset}/{p.name}"
        out_lbl = OUT_DIR / f"labels/{subset}/{p.stem}.txt"
        out_img.parent.mkdir(parents=True, exist_ok=True)
        out_lbl.parent.mkdir(parents=True, exist_ok=True)
 
        try:
            pil_img = Image.open(p).convert("RGB")
        except Exception as e:
            print(f"⚠️ Проблема с файлом {p}: {e}")
            continue

        W, H = pil_img.size
 
        results = detector(pil_img, candidate_labels=["tool", "instrument", "hand tool"])
 
        if not results:
            x, y, w, h = 0, 0, W, H
        else:
            r = max(results, key=lambda r: r["score"])
            x, y, w, h = (
                r["box"]["xmin"], r["box"]["ymin"],
                r["box"]["xmax"] - r["box"]["xmin"],
                r["box"]["ymax"] - r["box"]["ymin"]
            )
 
        cls_id = extract_class_id_from_folder(cls)
 
        line = f"{cls_id} {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}\n"

        shutil.copy2(p, out_img)
        with open(out_lbl, "w") as f:
            f.write(line)
 
        if i % PREVIEW_EVERY == 0:
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.imshow(pil_img)
            ax.add_patch(plt.Rectangle(
                (int(x), int(y)), int(w), int(h),
                edgecolor='lime', facecolor='none', linewidth=2
            ))
            ax.text(int(x), max(0, int(y) - 5), cls, color='lime', fontsize=12, weight='bold')
            ax.set_axis_off()
            ax.set_title(f"Прогресс: {cls} [{i + 1}/{len(paths)}]")
            preview_path = PREVIEW_DIR / f"{cls}_{i}.jpg"
            plt.savefig(preview_path, bbox_inches="tight")
            plt.close(fig)

print("✅ Разметка завершена. Превью сохранены в:", PREVIEW_DIR)

9 Открывашка для банок с маслом: 100%|██████████| 248/248 [39:22<00:00,  9.53s/it]    

✅ Разметка завершена. Превью сохранены в: C:\Users\ROG\Desktop\hachaton_model\datasets\yolo_dataset\preview


In [ ]:
# ====== Создание синтетических данных с помощью вырезок из сегментации ======
from pathlib import Path
import json, random, math
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

import torch
from segment_anything import sam_model_registry, SamPredictor
 
YOLO_IMG_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\yolo_dataset\images\train")
YOLO_LBL_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\yolo_dataset\labels\train")
CROPS_DIR     = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\crop_bank")   # кеш с альфой
META_JSON     = CROPS_DIR / "meta.json"

SYN_IMG_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\synthetic\images")
SYN_LBL_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\synthetic\labels")

for p in [CROPS_DIR, SYN_IMG_DIR, SYN_LBL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
 
CANVAS_W, CANVAS_H = 5152, 3864
SCALE_RANGE = (0.22, 0.48)
MAX_OBJECTS = 11
MAX_IOU     = 0.60
 
SAM_TYPE       = "vit_b"
SAM_CHECKPOINT = r"C:\Users\ROG\Desktop\hachaton_model\sam_vit_b_01ec64.pth"  
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

 
def load_yolo_labels(lbl_path, W, H):
    boxes = []
    with open(lbl_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, cx, cy, w, h = map(float, parts)
            x1 = int((cx - w/2) * W); y1 = int((cy - h/2) * H)
            x2 = int((cx + w/2) * W); y2 = int((cy + h/2) * H)
            boxes.append((int(cls), max(0,x1), max(0,y1), min(W-1,x2), min(H-1,y2)))
    return boxes

def bbox_from_mask(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max())+1, int(ys.max())+1

def iou_xyxy(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    if inter == 0: return 0.0
    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])
    return inter / (areaA + areaB - inter)

# ====== СТРОИМ БАНК ВЫРЕЗОК С АЛЬФОЙ ======
def build_crop_bank():
    sam = sam_model_registry[SAM_TYPE](checkpoint=SAM_CHECKPOINT).to(DEVICE)
    predictor = SamPredictor(sam)

    meta = []
    img_files = [p for p in YOLO_IMG_DIR.glob("*") if p.suffix in IMG_EXTS]
    for img_path in tqdm(img_files, desc="SAM crop bank"):
        lbl_path = YOLO_LBL_DIR / (img_path.stem + ".txt")
        if not lbl_path.exists(): 
            continue

        img_rgb = np.array(Image.open(img_path).convert("RGB"))
        H, W = img_rgb.shape[:2]
        boxes = load_yolo_labels(lbl_path, W, H)
        if not boxes:
            continue

        predictor.set_image(img_rgb)

        for k, (cls_id, x1, y1, x2, y2) in enumerate(boxes):
            box = np.array([x1, y1, x2, y2], dtype=np.float32)
            masks, scores, _ = predictor.predict(box=box, multimask_output=True)
            if masks is None or len(masks) == 0:
                mask = np.zeros((H,W), np.uint8)
                mask[y1:y2, x1:x2] = 255
            else:
                m = masks[scores.argmax()].astype(np.uint8) * 255
                mask = m

            bb = bbox_from_mask(mask)
            if bb is None:
                continue
            bx1, by1, bx2, by2 = bb
            crop_rgb  = img_rgb[by1:by2, bx1:bx2]
            crop_mask = mask[by1:by2, bx1:bx2]

            rgba = np.dstack([crop_rgb, crop_mask])
            out_png = CROPS_DIR / f"{img_path.stem}_{k:02d}__cls{cls_id}.png"
            Image.fromarray(rgba).save(out_png)
 
            cls_name = str(cls_id)    
            meta.append({"file": out_png.name, "cls": cls_name})
    
    with open(META_JSON, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False)
    print(f"✅ Кеш готов: {len(meta)} вырезок → {CROPS_DIR}")

# ====== СБОРКА СИНТЕТИКИ ======
def place_rgba_on_canvas(canvas, crop_rgba, x, y):
    h, w = crop_rgba.shape[:2]
    if x < 0 or y < 0 or x+w > canvas.shape[1] or y+h > canvas.shape[0]:
        return None
    rgb = crop_rgba[..., :3]
    a   = crop_rgba[..., 3:4].astype(np.float32)/255.0
    roi = canvas[y:y+h, x:x+w].astype(np.float32)
    comp = a * rgb + (1.0 - a) * roi
    canvas[y:y+h, x:x+w] = comp.astype(np.uint8)

    ys, xs = np.where(crop_rgba[...,3] > 0)
    if len(xs)==0 or len(ys)==0:
        return None
    x1 = x + int(xs.min()); x2 = x + int(xs.max()) + 1
    y1 = y + int(ys.min()); y2 = y + int(ys.max()) + 1
    return (x1, y1, x2, y2)

def yolo_line_from_xyxy(cls, box, W, H):
    x1,y1,x2,y2 = box
    w = x2 - x1; h = y2 - y1
    cx = (x1 + w/2) / W
    cy = (y1 + h/2) / H
    nw = w / W
    nh = h / H
    return f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n"

def make_synthetic_scene(meta, n_objects=None):
    Wc, Hc = CANVAS_W, CANVAS_H
    canvas = np.ones((Hc, Wc, 3), dtype=np.uint8) * 255
    labels, placed = [], []

    if n_objects is None:
        n_objects = random.randint(2, MAX_OBJECTS)
    chosen = random.sample(meta, min(n_objects, len(meta)))

    for item in chosen:
        cls = item["cls"]
        fp  = CROPS_DIR / item["file"]
        rgba = np.array(Image.open(fp).convert("RGBA"))
        h0, w0 = rgba.shape[:2]

        scale = random.uniform(*SCALE_RANGE)
        new_w = max(8, int(w0 * scale))
        new_h = max(8, int(h0 * scale))
        rgba_res = np.array(Image.fromarray(rgba).resize((new_w, new_h), Image.LANCZOS))

        new_w = min(new_w, Wc-8); new_h = min(new_h, Hc-8)
        rgba_res = rgba_res[:new_h, :new_w]

        ok = False
        for _ in range(40):
            x = random.randint(0, Wc - new_w)
            y = random.randint(0, Hc - new_h)
            trial_box = (x, y, x+new_w, y+new_h)
            if all(iou_xyxy(trial_box, pb) <= MAX_IOU for pb in placed):
                box = place_rgba_on_canvas(canvas, rgba_res, x, y)
                if box is None:
                    continue
                if all(iou_xyxy(box, pb) <= MAX_IOU for pb in placed):
                    placed.append(box)
                    labels.append(yolo_line_from_xyxy(cls, box, Wc, Hc))
                    ok = True
                    break
        if not ok:
            continue

    return canvas, labels

def generate_synthetic_dataset(n_images=300):
    if not META_JSON.exists():
        print("⚠️ Нет кеша вырезок. Сначала выполняем build_crop_bank().")
        build_crop_bank()

    meta = json.loads(Path(META_JSON).read_text(encoding="utf-8"))
    print(f"🧩 В банке {len(meta)} вырезок. Генерю сцены...")
    for i in tqdm(range(n_images), desc="Synth"):
        img, lbls = make_synthetic_scene(meta)
        out_img = SYN_IMG_DIR / f"synthetic_{i:04d}.jpg"
        out_lbl = SYN_LBL_DIR / f"synthetic_{i:04d}.txt"
        cv2.imwrite(str(out_img), img)
        with open(out_lbl, "w", encoding="utf-8") as f:
            f.writelines(lbls)
    print("✅ Готово:", SYN_IMG_DIR)

 
# создаём банк вырезок
build_crop_bank()

# Генерим синтетику
# generate_synthetic_dataset(n_images=300)


Device: cuda


c:\Users\ROG\Desktop\hachaton_model\.venv\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)
SAM c

✅ Кеш готов: 4141 вырезок → C:\Users\ROG\Desktop\hachaton_model\datasets\crop_bank


In [ ]:
# ====== Создание синтетических данных взвешенных по классам ======
from pathlib import Path
import json, random
import numpy as np
import cv2
from PIL import Image, ImageEnhance
from tqdm import tqdm
import yaml

# ---------- PATHS ----------
CROPS_DIR  = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\crop_bank")
META_JSON  = CROPS_DIR / "meta.json"

# ---------- CLASS WEIGHTS ----------
CLASS_WEIGHTS = {
    "Отвертка_минус": 3.0,
    "Отвертка_плюс": 3.0,
    "Отвертка_смещенный_крест": 3.0,
    "Пассатижи": 2.0,
    "Открывашка": 2.0,
    # остальные классы → вес 1.0
}

SYN_IMG_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\synthetic_aug_v3\images")
SYN_LBL_DIR = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\synthetic_aug_v3\labels")
for p in [SYN_IMG_DIR, SYN_LBL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

 
DATASET_YAML = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\yolo_dataset\dataset.yaml")

 
CANVAS_W, CANVAS_H = 5152, 3864
SCALE_RANGE = (0.22, 0.48)    
ROW_SCALE   = (0.20, 0.30)    
MAX_OBJECTS = 11
MAX_IOU     = 0.60            

 
def load_class2id():
    """Пытаемся взять словарь имён классов из dataset.yaml. Если нет — пустой."""
    if DATASET_YAML.exists():
        with open(DATASET_YAML, "r", encoding="utf-8") as f:
            names = yaml.safe_load(f).get("names", [])
        if isinstance(names, list):
            return {name: i for i, name in enumerate(names)}
    return {}

def extract_class_id(cls_field, class2id):
    """
    meta.json может хранить:
      - число (0..N-1) → возвращаем как int
      - строку с названием → маппим через class2id
      - строку, начинающуюся с числа → берём число
    """
    s = str(cls_field)
    first = s.split()[0]
    if first.isdigit():
        return int(first)
    return class2id.get(s, 0)

def iou_xyxy(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    if inter <= 0:
        return 0.0
    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])
    return inter / (areaA + areaB - inter)

def random_background(W, H):
    """Белый, серый или вертикальный градиент + лёгкий шум."""
    mode = random.choice(["white", "gray", "gradient"])
    if mode == "white":
        canvas = np.full((H, W, 3), 255, np.uint8)
    elif mode == "gray":
        val = random.randint(185, 230)
        canvas = np.full((H, W, 3), val, np.uint8)
    else:
        start = random.randint(200, 255)
        end   = random.randint(150, 220)
        grad = np.linspace(start, end, H, dtype=np.uint8)
        canvas = np.repeat(grad[:, None], W, axis=1)
        canvas = np.stack([canvas, canvas, canvas], axis=2)

 
    noise = np.random.normal(0, 6, (H, W, 3)).astype(np.float32)
    out = np.clip(canvas.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return out

def augment_rgba(rgba_np):
    """Поворот ±15°, небольшая яркость/контраст, лёгкий blur иногда."""
    img = Image.fromarray(rgba_np)
    angle = random.uniform(-15, 15)
    img = img.rotate(angle, expand=True, resample=Image.BICUBIC)

    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.9, 1.1))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.9, 1.12))

    rgba_np = np.array(img)
    if random.random() < 0.25:
        k = random.choice([3, 5])
        rgba_np[..., :3] = cv2.GaussianBlur(rgba_np[..., :3], (k, k), 0)
    return rgba_np

def place_rgba_on_canvas(canvas, crop_rgba, x, y):
    h, w = crop_rgba.shape[:2]
    if x < 0 or y < 0 or x + w > canvas.shape[1] or y + h > canvas.shape[0]:
        return None
    rgb = crop_rgba[..., :3]
    a   = (crop_rgba[..., 3:4].astype(np.float32) / 255.0)
    roi = canvas[y:y+h, x:x+w].astype(np.float32)
    comp = a * rgb + (1.0 - a) * roi
    canvas[y:y+h, x:x+w] = comp.astype(np.uint8)
    ys, xs = np.where(crop_rgba[..., 3] > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x1 = x + int(xs.min()); x2 = x + int(xs.max()) + 1
    y1 = y + int(ys.min()); y2 = y + int(ys.max()) + 1
    return (x1, y1, x2, y2)

def yolo_line_from_xyxy(cls, box, W, H):
    x1, y1, x2, y2 = box
    w = x2 - x1; h = y2 - y1
    cx = (x1 + w / 2) / W
    cy = (y1 + h / 2) / H
    nw = w / W
    nh = h / H
    return f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n"

# ---------- CORE ----------
def make_synthetic_scene(meta, class2id, n_objects=None, mode="random"):
    Wc, Hc = CANVAS_W, CANVAS_H
    canvas = random_background(Wc, Hc)
    labels, placed = [], []

    if mode == "crowd":
        n_objects = random.randint(12, 20)
        chosen = random.sample(meta, min(n_objects, len(meta)))

    elif mode == "weighted":
        n_objects = random.randint(5, 14)

         
        weights = []
        for item in meta:
            cls_name = str(item["cls"])
            w = CLASS_WEIGHTS.get(cls_name, 1.0)
            weights.append(w)

 
        chosen = random.choices(meta, weights=weights, k=n_objects)

    elif mode == "row":
        classes = sorted(set(item["cls"] for item in meta))
        chosen = []
        for cls in classes:
            candidates = [m for m in meta if m["cls"] == cls]
            if candidates:
                chosen.append(random.choice(candidates))

        chosen = sorted(chosen, key=lambda x: x["cls"])
        margin = 20
        x_cursor = margin
        y_base = Hc // 2

        for item in chosen:
            cls_id = extract_class_id(item["cls"], class2id)
            fp = CROPS_DIR / item["file"]
            if not fp.exists():    
                continue
            rgba = np.array(Image.open(fp).convert("RGBA"))

            h0, w0 = rgba.shape[:2]
            scale = random.uniform(0.2, 0.3)
            new_w = max(8, int(w0 * scale))
            new_h = max(8, int(h0 * scale))
            rgba_res = np.array(Image.fromarray(rgba).resize((new_w, new_h), Image.LANCZOS))

            y = max(0, y_base - new_h // 2)
            x = x_cursor

            box = place_rgba_on_canvas(canvas, rgba_res, x, y)
            if box is not None:
                labels.append(yolo_line_from_xyxy(cls_id, box, Wc, Hc))
                placed.append(box)

            x_cursor += int(new_w * random.uniform(0.85, 0.95)) + margin

        return canvas, labels
    

    else:   
        if n_objects is None:
            n_objects = random.randint(2, MAX_OBJECTS)
        chosen = random.sample(meta, min(n_objects, len(meta)))

  
    for item in chosen:
        cls_id = extract_class_id(item["cls"], class2id)
        fp = CROPS_DIR / item["file"]
        if not fp.exists():    
            continue
        rgba = np.array(Image.open(fp).convert("RGBA"))

        h0, w0 = rgba.shape[:2]
        scale = random.uniform(*SCALE_RANGE)
        new_w = max(8, int(w0 * scale))
        new_h = max(8, int(h0 * scale))
        rgba_res = np.array(Image.fromarray(rgba).resize((new_w, new_h), Image.LANCZOS))

        ok = False
        for _ in range(40):
            x = random.randint(0, Wc - new_w)
            y = random.randint(0, Hc - new_h)
            trial_box = (x, y, x+new_w, y+new_h)
            if all(iou_xyxy(trial_box, pb) <= MAX_IOU for pb in placed):
                box = place_rgba_on_canvas(canvas, rgba_res, x, y)
                if box is not None:
                    placed.append(box)
                    labels.append(yolo_line_from_xyxy(cls_id, box, Wc, Hc))
                    ok = True
                    break
        if not ok:
            continue

    return canvas, labels

def generate_synthetic_dataset(n_images=50, mode="random", class2id=None, prefix=None):
    """
    n_images: сколько картинок сгенерить
    mode: "random" | "crowd" | "row"
    class2id: словарь имя->id; если None, попытаемся прочитать из dataset.yaml
    prefix: префикс в имени файлов (по умолчанию = mode)
    """
    if not META_JSON.exists():
        raise RuntimeError("❌ Нет кеша вырезок: meta.json не найден. Сначала сделай crop_bank.")

    if class2id is None:
        class2id = load_class2id()

    meta = json.loads(META_JSON.read_text(encoding="utf-8"))
    print(f"🧩 Выборок в банке: {len(meta)} | режим: {mode}")

    tag = prefix if isinstance(prefix, str) and prefix else mode
    created = 0
    for i in tqdm(range(n_images), desc=f"Synth {mode}"):
        img, lbls = make_synthetic_scene(meta, class2id, mode=mode)
        if not lbls:
            continue  # пропустим пустые сцены
        out_img = SYN_IMG_DIR / f"{tag}_{i:04d}.jpg"
        out_lbl = SYN_LBL_DIR / f"{tag}_{i:04d}.txt"
        cv2.imwrite(str(out_img), img)
        with open(out_lbl, "w", encoding="utf-8") as f:
            f.writelines(lbls)
        created += 1

    print(f"✅ Создано сцен: {created}. Смотри: {SYN_IMG_DIR}")

# --------- Вызовы ---------
# CLASS2ID = load_class2id()   # или свой словарь, если не используешь dataset.yaml

# # Random сцены
# generate_synthetic_dataset(n_images=2000, mode="random", class2id=None)

# # Crowd сцены (много объектов, умеренное перекрытие)
# generate_synthetic_dataset(n_images=700, mode="crowd", class2id=None)

# # Row сцены (по одному объекту каждого класса в 1–2 ряда)
# generate_synthetic_dataset(n_images=1500, mode="row", class2id=None)

In [18]:
# # === Random сцены ===
# generate_synthetic_dataset(n_images=2000, mode="random", class2id=None)

# # === Crowd сцены ===
# generate_synthetic_dataset(n_images=700, mode="crowd", class2id=None)
# === Row сцены ===
# generate_synthetic_dataset(n_images=1800, mode="row", class2id=None)

generate_synthetic_dataset(n_images=500, mode="weighted", class2id=None)

🧩 Выборок в банке: 4141 | режим: weighted


Synth weighted: 100%|██████████| 500/500 [18:55<00:00,  2.27s/it]

✅ Создано сцен: 500. Смотри: C:\Users\ROG\Desktop\hachaton_model\datasets\synthetic_aug_v3\images


In [ ]:
# Слияние синтетических и чистых данных

import random
import shutil
from pathlib import Path
from tqdm import tqdm


YOLO_DATASET = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\yolo_dataset")
SYNTHETIC_DIRS = [
    Path(r"C:\Users\ROG\Desktop\hachaton_model\test_dataset"),
 
]

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_images(folder: Path):
    """Возвращает список всех изображений в папке"""
    return [p for p in folder.glob("*") if p.suffix.lower() in IMG_EXTS]

def find_label_for_image(img_path: Path, labels_dir: Path):
    """Находит соответствующий файл разметки для изображения"""
    for ext in IMG_EXTS:
        lbl_path = labels_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            return lbl_path
    return None

def clean_orphan_labels(img_dir: Path, lbl_dir: Path):
    """Удаляет txt без картинки"""
    removed = 0
    for lbl in lbl_dir.glob("*.txt"):
        img_candidates = [img_dir / (lbl.stem + ext) for ext in IMG_EXTS]
        if not any(img.exists() for img in img_candidates):
            lbl.unlink()
            removed += 1
    if removed:
        print(f"🗑 Удалено {removed} лишних разметок из {lbl_dir.name}")

def copy_with_shuffling(existing_imgs, new_pairs, dst_img_dir: Path, dst_lbl_dir: Path, split_name: str):
    """Копирует файлы с полным перемешиванием существующих и новых данных"""
    
   
    temp_dir = Path("temp_shuffle")
    temp_img_dir = temp_dir / "images" / split_name
    temp_lbl_dir = temp_dir / "labels" / split_name
    temp_img_dir.mkdir(parents=True, exist_ok=True)
    temp_lbl_dir.mkdir(parents=True, exist_ok=True)
    
 
    print(f"📥 Копируем существующие {split_name} файлы...")
    existing_pairs = []
    for img_path in tqdm(existing_imgs, desc=f"Существующие {split_name}"):
        lbl_path = find_label_for_image(img_path, YOLO_DATASET / "labels" / split_name)
        if lbl_path and lbl_path.exists():
        
            new_name = f"existing_{img_path.name}"
            new_img_path = temp_img_dir / new_name
            new_lbl_path = temp_lbl_dir / (img_path.stem + ".txt")
            
            shutil.copy2(img_path, new_img_path)
            shutil.copy2(lbl_path, new_lbl_path)
            existing_pairs.append((new_img_path, new_lbl_path))
    
  
    print(f"📥 Копируем новые {split_name} файлы...")
    new_copied_pairs = []
    for img_path, lbl_path in tqdm(new_pairs, desc=f"Новые {split_name}"):
        new_name = f"new_{img_path.name}"
        new_img_path = temp_img_dir / new_name
        new_lbl_path = temp_lbl_dir / (img_path.stem + ".txt")
        
        shutil.copy2(img_path, new_img_path)
        shutil.copy2(lbl_path, new_lbl_path)
        new_copied_pairs.append((new_img_path, new_lbl_path))
    
   
    all_pairs = existing_pairs + new_copied_pairs
    random.shuffle(all_pairs)
    print(f"🔄 Перемешиваем {len(all_pairs)} файлов в {split_name}...")
    
   
    for file in dst_img_dir.glob("*"):
        if file.is_file():
            file.unlink()
    for file in dst_lbl_dir.glob("*"):
        if file.is_file():
            file.unlink()
    
     
    print(f"📤 Копируем перемешанные {split_name} файлы...")
    for i, (img_path, lbl_path) in enumerate(tqdm(all_pairs, desc=f"Перемешанные {split_name}")):
        
        new_img_name = f"{i:06d}{img_path.suffix}"
        new_lbl_name = f"{i:06d}.txt"
        
        dst_img = dst_img_dir / new_img_name
        dst_lbl = dst_lbl_dir / new_lbl_name
        
        shutil.copy2(img_path, dst_img)
        shutil.copy2(lbl_path, dst_lbl)
    
    
    shutil.rmtree(temp_dir)
    
    return len(new_copied_pairs)

# === ОСНОВНОЙ КОД ===

train_imgs = list_images(YOLO_DATASET / "images" / "train")
val_imgs   = list_images(YOLO_DATASET / "images" / "val")

total = len(train_imgs) + len(val_imgs)
train_ratio = len(train_imgs) / total if total > 0 else 0.85
print(f"⚖️ Текущее Train/Val соотношение: {train_ratio:.2f} / {1-train_ratio:.2f}")
print(f"📊 Файлов: train={len(train_imgs)}, val={len(val_imgs)}")


all_synth = []
for synth_dir in SYNTHETIC_DIRS:
    img_dir = synth_dir / "images"
    lbl_dir = synth_dir / "labels"
    if not img_dir.exists() or not lbl_dir.exists():
        continue
    for img in list_images(img_dir):
        lbl = lbl_dir / (img.stem + ".txt")
        if lbl.exists():
            all_synth.append((img, lbl))

print(f"📦 Найдено синтетики (img+lbl): {len(all_synth)}")
random.shuffle(all_synth)


split_idx = int(len(all_synth) * train_ratio)
train_synth = all_synth[:split_idx]
val_synth   = all_synth[split_idx:]

print(f"📋 Новые данные: train={len(train_synth)}, val={len(val_synth)}")

train_copied = copy_with_shuffling(
    train_imgs, train_synth, 
    YOLO_DATASET / "images" / "train", 
    YOLO_DATASET / "labels" / "train",
    "train"
)

val_copied = copy_with_shuffling(
    val_imgs, val_synth,
    YOLO_DATASET / "images" / "val",
    YOLO_DATASET / "labels" / "val", 
    "val"
)

print(f"✅ Добавлено: {train_copied} в train, {val_copied} в val")
print(f"🏁 Итоговый датасет: {YOLO_DATASET}")

final_train = len(list_images(YOLO_DATASET / "images" / "train"))
final_val = len(list_images(YOLO_DATASET / "images" / "val"))
print(f"📊 Итоговые файлы: train={final_train}, val={final_val}")

⚖️ Текущее Train/Val соотношение: 0.85 / 0.15
📊 Файлов: train=6630, val=1191
📦 Найдено синтетики (img+lbl): 198
📋 Новые данные: train=167, val=31
📥 Копируем существующие train файлы...


Существующие train: 100%|██████████| 6630/6630 [01:26<00:00, 76.70it/s] 


📥 Копируем новые train файлы...


Новые train: 100%|██████████| 167/167 [00:01<00:00, 165.12it/s]


🔄 Перемешиваем 6797 файлов в train...
📤 Копируем перемешанные train файлы...


...

📥 Копируем новые val файлы...


Новые val: 100%|██████████| 31/31 [00:01<00:00, 29.76it/s]


🔄 Перемешиваем 1222 файлов в val...
📤 Копируем перемешанные val файлы...


Перемешанные val: 100%|██████████| 1222/1222 [00:37<00:00, 32.90it/s]


✅ Добавлено: 167 в train, 31 в val
🏁 Итоговый датасет: C:\Users\ROG\Desktop\hachaton_model\datasets\yolo_dataset
📊 Итоговые файлы: train=6797, val=1222


In [ ]:
# Формирование yaml файла для обучения

import yaml
from pathlib import Path

YOLO_DATASET = Path(r"C:\Users\ROG\Desktop\hachaton_model\datasets\yolo_dataset")

CLASSES = [
    "Отвертка_минус",
    "Отвертка_плюс",
    "Отвертка_смещенный_крест",
    "Коловорот",
    "Пассатижи_контровочные",
    "Пассатижи",
    "Шэрница",
    "Разводной_ключ",
    "Открывашка",
    "Ключ_рожковый_накидной_3_4",
    "Бокорезы"
]

dataset_yaml = {
    "train": str((YOLO_DATASET / "images" / "train").resolve()),
    "val": str((YOLO_DATASET / "images" / "val").resolve()),
    "nc": len(CLASSES),
    "names": CLASSES
}

out_path = YOLO_DATASET / "dataset.yaml"
with open(out_path, "w", encoding="utf-8") as f:
    yaml.dump(dataset_yaml, f, allow_unicode=True)

print("✅ YAML сохранён:", out_path)


✅ YAML сохранён: C:\Users\ROG\Desktop\hachaton_model\datasets\yolo_dataset\dataset.yaml


In [ ]:
# Обучение и дообучение(на тестовых размеченнхы наборах инструментов)

from ultralytics import YOLO
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = YOLO(r"C:\Users\ROG\Desktop\hachaton_model\scripts\runs\detect\new_data_roboflow\weights\best.pt")    

results = model.train(
    data=r"C:\Users\ROG\Desktop\hachaton_model\final_train_dataset\conf.yaml",  
    epochs=40,          
    imgsz=1280,           
    batch=8,            
    device=0,               
    name="final_training_40",   
    optimizer="AdamW",
    lr0=0.001,            
    cache="ram",
    freeze=10,
    mosaic=0.0,
    mixup=0.0,
    workers=0        
)


🔥 Обучаем на: cuda
Ultralytics 8.3.203  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ROG\Desktop\hachaton_model\final_train_dataset\conf.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\ROG\Desktop\hachaton_model\scripts\runs\detect\new_data_roboflow\weights\best.pt, momentum=0.937, mosaic=0.0, multi_sca

In [1]:
# Прогон модели

from ultralytics import YOLO


model = YOLO(r"C:\Users\ROG\Desktop\hachaton_model\scripts\runs\detect\new_data\weights\best.pt")

results = model.predict(
    source=r"C:\Users\ROG\Desktop\hachaton_model\scripts\test_images\DSCN5000.JPG", 
    imgsz=1024,    
    conf=0.5,     
    iou=0.4,       
    max_det=300,   
    save=True,     
    save_txt=True, 
    save_conf=True 
)

print("Прогон завершён. Результаты в папке:", model.predictor.save_dir)



image 1/1 C:\Users\ROG\Desktop\hachaton_model\scripts\test_images\DSCN5000.JPG: 768x1024 1 _, 1 _, 1 __, 1 , 1 _, 1 , 2 s, 49.3ms
Speed: 10.5ms preprocess, 49.3ms inference, 198.9ms postprocess per image at shape (1, 3, 768, 1024)
Results saved to C:\Users\ROG\Desktop\runs\detect\predict
1 label saved to C:\Users\ROG\Desktop\runs\detect\predict\labels
Прогон завершён. Результаты в папке: C:\Users\ROG\Desktop\runs\detect\predict
